# micro.py — 15-Min Bar Move Timing (Walk-Forward)

**Purpose:** For SPY 15-minute bars over the last 60 trading days, find bars
where volume and volatility are both elevated (a "signal"), then walk
forward in time from each signal to measure: how many bars it took for
price to move 2 standard deviations away from the signal price, what the
volume and volatility looked like at the moment of the signal, and what the
resulting percent price move was. Walk-forward by construction — every
rolling stat only looks backward at each point in time.

**Data:** SPY 15-minute OHLCV via yfinance, free — 60 days ending
2026-08-14 (Friday), matching yfinance's free intraday history limit.

**Design choices:**
- "Volatility" = rolling standard deviation of 15-min log returns over the
  trailing 20 bars (~5 hours), recomputed at every bar (no lookahead).
- "2 std move" = 2x that bar's trailing volatility, in log-return terms.
- Signal condition = volume AND volatility both above their own 60-day
  median at that point — a rough, adjustable threshold.
- Output per signal: bars-to-move, volume-at-signal, volatility-at-signal,
  and % move achieved.

**Fix note:** recent yfinance versions return a MultiIndex on columns even
for a single ticker. Both cells flatten columns immediately after download.

**Total code: ~50 lines across 2 cells.**


In [4]:
# --- Block 1: fetch 15-min bars, compute rolling volume + volatility ---
import yfinance as yf
import pandas as pd
import numpy as np

# 60 days of 15-min SPY bars, ending Friday 2026-08-14 (yfinance's free cap)
spy = yf.download("SPY", period="60d", interval="15m")

# Flatten MultiIndex columns (yfinance returns (field, ticker) pairs)
if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)
spy = spy.loc[:, ~spy.columns.duplicated()]  # guard against any duplicate columns
spy = spy[["Open", "High", "Low", "Close", "Volume"]].dropna()

# 15-min log returns
spy["Return"] = np.log(spy["Close"] / spy["Close"].shift(1))

# Rolling (backward-looking only) volume and volatility, ~5 hours = 20 bars
window = 20
spy["VolRoll"] = spy["Volume"].rolling(window).mean()
spy["VolatilityRoll"] = spy["Return"].rolling(window).std()

# Signal: both volume and volatility above their own median for this window
vol_thresh = spy["VolRoll"].median()
volt_thresh = spy["VolatilityRoll"].median()
spy["Signal"] = (spy["VolRoll"] > vol_thresh) & (spy["VolatilityRoll"] > volt_thresh)

print(f"Total 15-min bars: {len(spy)}")
print(f"Signal bars flagged: {int(spy['Signal'].sum())}")


[*********************100%***********************]  1 of 1 completed

Total 15-min bars: 1560
Signal bars flagged: 556


In [8]:
# --- Block 2: walk forward from each signal, measure bars-to-2std-move ---
results = []
max_lookahead = 40  # cap search at 40 bars (~10 hours) forward

for sig_time in spy.index[spy["Signal"]]:
    idx = spy.index.get_loc(sig_time)
    sigma = float(spy["VolatilityRoll"].iloc[idx])
    entry_price = float(spy["Close"].iloc[idx])
    target_move = 2 * sigma  # 2-std threshold, in log-return terms

    for j in range(idx + 1, min(idx + max_lookahead, len(spy))):
        cum_return = float(np.log(spy["Close"].iloc[j] / entry_price))
        if abs(cum_return) >= target_move:
            results.append({
                "bars_to_move": j - idx,
                "volume_at_signal": float(spy["VolRoll"].iloc[idx]),
                "volatility_at_signal": sigma,
                "pct_move": cum_return * 100,
            })
            break

results_df = pd.DataFrame(results)
print(f"Signals that reached a 2-std move within {max_lookahead} bars: {len(results_df)}")
if len(results_df) > 0:
    print(results_df.describe())
else:
    print("No signals resolved in time. Try a wider max_lookahead or a looser Signal condition.")


Signals that reached a 2-std move within 40 bars: 517
       bars_to_move  volume_at_signal  volatility_at_signal    pct_move
count    517.000000      5.170000e+02            517.000000  517.000000
mean      12.390716      2.142802e+06              0.002156    0.060234
std        8.848759      4.617873e+05              0.000700    0.647146
min        1.000000      1.573308e+06              0.001252   -1.579429
25%        5.000000      1.794084e+06              0.001555   -0.489166
50%       10.000000      2.013898e+06              0.002072    0.324150
75%       19.000000      2.329911e+06              0.002457    0.576567
max       39.000000      3.994205e+06              0.004012    1.870368
